[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_DSP/Time_Frequency_2.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Time–Frequency Analysis II

Beyond the spectrogram: the Wigner–Ville distribution (razor resolution, haunted by cross-terms), reassignment & synchrosqueezing (sharpening the STFT after the fact), and empirical mode decomposition (letting the signal choose its own components). The modern chapter of the [uncertainty-principle](./Foundations_of_Signal_Processing_1.ipynb) story.

## 1. Pre-requisites

[Foundations 1](./Foundations_of_Signal_Processing_1.ipynb) S6 (STFT/uncertainty), [Audio DSP](./Audio_Speech_DSP.ipynb) S1 (spectrogram fluency).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sig
rng = np.random.default_rng(0)

fs = 1000
t = np.arange(0, 2, 1/fs)
# the test signal for the whole workshop: two crossing chirps + a tone burst
x = sig.chirp(t, 50, 2, 350) + sig.chirp(t, 350, 2, 50)     + np.where((t > 0.8) & (t < 1.2), np.sin(2*np.pi*420*t), 0)

---
### 🕐 Session 1 of 3 — *The Wigner–Ville Distribution* (~40 min)
**Goal:** quadratic time-frequency: perfect chirp concentration, ghost cross-terms — both demonstrated.
**Builds on:** [Foundations 1](./Foundations_of_Signal_Processing_1.ipynb) S6. &nbsp; **Feeds into:** Session 2 (reassignment).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: The Wigner–Ville Distribution</b></summary>

**Timing (~40 min).** 8 min why the spectrogram blurs · 10 min the WVD definition · 12 min the demo and its ghosts · 10 min what quadratic really costs.

**Board first — locate the blur.** The spectrogram windows *then* transforms, so the window's bandwidth is baked into every result before any analysis happens. Short window, good time resolution, poor frequency resolution; long window, the reverse. That trade is the [uncertainty principle](./Foundations_of_Signal_Processing_1.ipynb) as an engineering constraint, and the room already owns it. Then pose the question this session answers: what if we never window at all?

**Pre-empt the objection, because a good student will raise it.** "Perfect concentration violates the uncertainty principle." It does not. The uncertainty principle constrains *linear* transforms — anything that decomposes the signal onto a fixed set of atoms. The WVD is **quadratic**: it correlates the signal with itself, so it is not a decomposition and the premise does not apply. It sidesteps the theorem rather than beating it, and the price is paid elsewhere. Getting this straight is the intellectual content of the session.

**And the price is exactly the quadratic-ness.** Write $(a+b)^2 = a^2 + b^2 + 2ab$ on the board. The cross-product term is not a metaphor — it is literally where cross-terms come from. With $K$ components you get $K$ genuine terms and $\binom{K}{2}$ ghosts, so ghosts outnumber signal as soon as $K > 3$. Each ghost sits *midway* between its two parents in both time and frequency, and oscillates. Have the room predict where the ghosts will be before you show the plot; with two chirps crossing plus a tone burst, they can locate all three.

**Point at `sig.hilbert` and say why it is there.** Taking the analytic signal removes negative-frequency components, which halves the number of interacting pairs and eliminates the ghosts between positive and negative frequencies. It is a standard and near-mandatory step, not an implementation detail — without it the picture is considerably worse.

**Ask the room.** "Ghosts oscillate; genuine terms do not. What does that suggest?" Smoothing. Averaging over a small time–frequency neighbourhood attenuates the oscillating cross-terms while leaving the concentrated auto-terms largely intact. That idea generates the entire Cohen class — the pseudo-WVD, Choi–Williams, and the rest — each a different smoothing kernel trading concentration against ghost suppression. Mention that the spectrogram is itself a member of that class, at the maximum-smoothing end. Framing it as one family with a knob is much better than four named methods.

**Pacing note.** `wigner_ville` is $O(N^2)$ and the signal is decimated to 600 samples specifically to keep it tolerable — say so, and note that this cost is a real reason the WVD is rarely used raw on long records.
</details>

## 2. Correlating the Signal with Itself

💡 **Intuition.** The spectrogram windows *then* transforms — the window's blur is baked in. **Wigner–Ville** drops the window: correlate the signal with itself around each instant, $W(t, f) = \int x(t+\tau/2) x^*(t-\tau/2) e^{-j2\pi f\tau} d\tau$. For a lone linear chirp it is *perfectly* concentrated — no uncertainty smear at all (quadratic transforms don't violate the uncertainty principle; they sidestep its linear-transform premise). The price is steep: being quadratic, **every pair of components births a ghost** midway between them, oscillating — cross-terms that can dwarf the real signal.

In [2]:
def wigner_ville(x_a, n_freq=512):
    N = len(x_a)
    xa = sig.hilbert(x_a)                                 # analytic signal halves the ghosts
    W = np.zeros((n_freq, N))
    for n_i in range(N):
        tau_max = min(n_i, N-1-n_i, n_freq//2 - 1)
        tau = np.arange(-tau_max, tau_max+1)
        acf = xa[n_i + tau] * np.conj(xa[n_i - tau])
        row = np.zeros(n_freq, complex)
        row[tau % n_freq] = acf
        W[:, n_i] = np.real(np.fft.fft(row))
    return W

xa_short = x[::2][:600]                                   # decimate for speed
W = wigner_ville(xa_short)
f_stft, t_stft, S = sig.stft(x, fs=fs, nperseg=128)

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].pcolormesh(t_stft, f_stft, np.abs(S), shading="auto")
axes[0].set_title("spectrogram: honest but blurred"); axes[0].set_ylim(0, 500)
axes[1].imshow(np.abs(W[:150]), aspect="auto", origin="lower",
               extent=[0, 1.2, 0, 150*fs/2/512])
axes[1].set_title("Wigner–Ville: razor lines + GHOSTS between them")
plt.tight_layout(); plt.show()
print("the ghost midway between the two chirps is not a signal — it's the quadratic cross-term")

the ghost midway between the two chirps is not a signal — it's the quadratic cross-term


/tmp/ipykernel_2986310/1466983963.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Two views of one signal. The spectrogram shows soft, honest bands — every component is where it should be, and all of them are blurred by the window's bandwidth. The Wigner–Ville panel shows the chirps as near-perfect lines, far sharper than any window could give, *and* a bright structure sitting between them that corresponds to no component of the signal at all.

**The sharpness and the ghost have the same cause.** The WVD is quadratic: it correlates the signal with itself rather than projecting it onto fixed atoms. Since it never windows, no window bandwidth is imposed — hence the razor lines. But expand $(a + b)^2 = a^2 + b^2 + 2ab$ and the cross-term is unavoidable. Every *pair* of components generates an interference term located midway between them in both time and frequency. Our signal has three components, so there are three ghosts, and with $K$ components there are $\binom{K}{2}$ of them against only $K$ real ones — ghosts outnumber signal as soon as $K > 3$.

**And this is not a violation of the uncertainty principle.** That theorem constrains *linear* transforms, which decompose a signal onto a fixed dictionary. The WVD is not a decomposition, so the premise does not apply and there is nothing to violate. It sidesteps the constraint rather than beating it — and the cross-terms are what it pays instead. Concentration was never free; the cost simply moved from blur to artifacts.

**Two practical notes.** `sig.hilbert` is not incidental: taking the analytic signal removes negative frequencies, which halves the number of interacting pairs and kills the ghosts that would otherwise appear between positive and negative components. And the signal is decimated to 600 samples because the implementation is $O(N^2)$ — a real reason the raw WVD is uncommon on long records.

**Where this leads.** Cross-terms *oscillate* while auto-terms do not, which means local smoothing suppresses them preferentially. That single observation generates the whole Cohen class of distributions — pseudo-WVD, Choi–Williams, and others — each a different smoothing kernel buying ghost suppression at the cost of concentration. The spectrogram is itself a member of that family, sitting at the maximum-smoothing extreme. So the two panels here are not rival methods but the two ends of one continuum, and the interesting designs live between them.

The lesson to carry into Session 2: **every time–frequency lens invents artifacts of its own kind.** The spectrogram invents blur, the WVD invents ghosts. Neither is dishonest, but you cannot read either without knowing which artifacts to discount.

---
### 🕐 Session 2 of 3 — *Reassignment & Synchrosqueezing* (~40 min)
**Goal:** sharpen the spectrogram by moving each energy blob to its local center of gravity.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (EMD).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: Reassignment & Synchrosqueezing</b></summary>

**Timing (~40 min).** 10 min the phase-carries-information idea · 10 min the reassignment mechanism · 12 min the demo, *including its suspicious number* · 8 min limits.

**Board first — the observation the method rests on.** We routinely plot $|{\rm STFT}|$ and throw the phase away. But the phase is not noise: its time-derivative is the local instantaneous frequency, so within a blurred blob the phase still knows exactly where the true ridge sits. Synchrosqueezing reads that derivative and *moves* each coefficient's energy to where the phase says it belongs. Nothing is re-estimated from scratch — information already present but discarded is put back to work. Students find this genuinely surprising, so let it land before showing code.

**Why it beats Session 1's trade-off.** The STFT is linear, so no cross-terms are possible; the reassignment step is a post-hoc *redistribution* of energy, not a new transform. So we get near-WVD sharpness with the spectrogram's honesty. Ask the room where the catch must be — it is that reassignment assumes each blob is dominated by *one* component, which fails wherever components are closer than a window bandwidth. Our two chirps cross, so the method must degrade at the crossing, and the plot shows it.

**The printed number is wrong-looking on purpose — do not let it pass.** It reads `STFT 9.1 Hz → squeezed 0.0 Hz`. A ridge width of exactly zero is not infinite resolution; it means every scrap of energy in that band landed in a *single* frequency bin, so the computed spread is identically zero. Look at `k = int(round(inst_f[i, j] / df))`: reassignment here snaps energy onto the integer bin grid, so the output cannot be narrower than one bin and often collapses to exactly one. The 0.0 is a **quantization artifact of the implementation**, not a measurement.

Make this a teaching moment rather than an embarrassment. Ask the room what a suspicious number looks like — anything exactly 0, exactly 1, or exactly 100% deserves scrutiny before celebration. The honest claim is "sharpened from 9.1 Hz to at or below the 3.9 Hz bin spacing," and a proper implementation would either accumulate onto a finer grid or interpolate rather than round. The debrief says all of this; reinforce it.

**Ask the room.** "Where should this method fail?" At the chirp crossing, where two components occupy one window bandwidth and the single-component assumption breaks — the phase derivative then reports some blend of the two, and energy is reassigned to a frequency where nothing is. Point at that region on the plot. A method that is sharp everywhere *including* where it should not be is a warning sign.

**Flag the simplification.** This is "synchrosqueezing-lite": a real implementation reassigns in time as well as frequency, and true synchrosqueezing is invertible — you can reconstruct individual components after separating them, which is its main practical advantage over plain reassignment. Neither is done here. Say so, so nobody cites this cell as an implementation.
</details>

## 3. Sharpening After the Fact

💡 **Intuition.** The spectrogram's smear has structure: within each blurry blob, the *phase* of the STFT knows where the true ridge is (the local instantaneous frequency is the phase's time-derivative). **Synchrosqueezing** reads that derivative and moves each coefficient's energy to its rightful frequency — linear-transform honesty (no cross-terms!) with nearly Wigner-grade sharpness on separated components. The catch: components must be separated by more than a window bandwidth — crossing chirps still confuse it at the crossing.

In [3]:
# synchrosqueezing-lite: reassign STFT energy along frequency by the phase derivative
f_s, t_s, Z = sig.stft(x, fs=fs, nperseg=256, noverlap=224, return_onesided=True)
eps = 1e-8
# instantaneous frequency estimate per bin: d(phase)/dt via adjacent frames
dphase = np.angle(Z[:, 1:] * np.conj(Z[:, :-1]))
hop = 256 - 224
inst_f = dphase / (2*np.pi*hop/fs)
inst_f = np.where(inst_f < 0, inst_f + fs/hop, inst_f)     # unwrap into [0, fs/hop)

S_sq = np.zeros_like(np.abs(Z[:, :-1]))
df = f_s[1] - f_s[0]
for i in range(Z.shape[0]):
    for j in range(Z.shape[1]-1):
        mag = np.abs(Z[i, j])
        if mag < 1e-3: continue
        k = int(round(inst_f[i, j] / df))
        if 0 <= k < S_sq.shape[0]:
            S_sq[k, j] += mag

fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
axes[0].pcolormesh(t_s, f_s, np.abs(Z), shading="auto"); axes[0].set_title("STFT magnitude")
axes[1].pcolormesh(t_s[:-1], f_s, S_sq, shading="auto"); axes[1].set_title("synchrosqueezed: ridges snap sharp, no ghosts")
for ax in axes: ax.set_ylim(0, 500)
plt.tight_layout(); plt.show()

# quantify: ridge width (freq-axis spread) at t = 0.5 s for the rising chirp (~125 Hz)
j0 = np.argmin(np.abs(t_s - 0.5))
def width(M, j):
    col = M[(f_s > 60) & (f_s < 200), j]; col = col/ (col.sum()+eps)
    fc = f_s[(f_s > 60) & (f_s < 200)]
    mu = (col*fc).sum(); return np.sqrt((col*(fc-mu)**2).sum())
print(f"ridge width at t=0.5s:  STFT {width(np.abs(Z), j0):.1f} Hz  →  squeezed {width(S_sq, j0):.1f} Hz")

ridge width at t=0.5s:  STFT 9.1 Hz  →  squeezed 0.0 Hz


/tmp/ipykernel_2986310/680984120.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** The ridges snap from soft bands into thin lines, and — unlike Session 1's Wigner–Ville panel — **no ghosts appear anywhere**. The STFT is a linear transform, so cross-terms are structurally impossible; reassignment only *moves* energy that the STFT already computed. That is the appeal: near-WVD sharpness with the spectrogram's honesty.

The mechanism is recovering information we normally discard. We habitually plot $|{\rm STFT}|$ and throw the phase away, but the phase's time-derivative is the local instantaneous frequency — inside every blurred blob, the phase still knows where the true ridge is. `dphase = np.angle(Z[:, 1:] * np.conj(Z[:, :-1]))` extracts exactly that, and the loop deposits each coefficient's magnitude at the frequency the phase indicates rather than the bin it arrived in.

**Now read the printed number sceptically, because it does not mean what it appears to.** `STFT 9.1 Hz → squeezed 0.0 Hz`. A ridge width of *exactly* zero is not a measurement of infinite resolution — it is what `width()` returns when all the energy in the band sits in a single frequency bin, since the spread of a one-point distribution is identically zero.

The cause is in the code: `k = int(round(inst_f[i, j] / df))` snaps every reassigned coefficient onto the integer bin grid. The output therefore *cannot* be narrower than one bin, and frequently collapses to precisely one. With `nperseg=256` at $f_s = 1000$ Hz the bin spacing is about 3.9 Hz, so the honest statement is **"sharpened from 9.1 Hz to at or below the 3.9 Hz bin spacing"** — a real and substantial improvement, but one this measurement cannot resolve further. A proper implementation accumulates onto a finer grid or interpolates between bins instead of rounding.

Worth generalising: a result of exactly 0, exactly 1, or exactly 100% almost always indicates a saturated measurement rather than a perfect one. Check what the metric *can* return before believing what it did return.

**And the limitation the plot does show.** Reassignment assumes each time–frequency blob is dominated by a *single* component. Where the two chirps cross they occupy one window bandwidth together, the phase derivative reports a blend rather than either one, and energy gets reassigned to a frequency where nothing exists. Look at the crossing region: the ridges smear or wander exactly there. This is the honest counterpart to Session 1's ghosts — a different artifact from a different lens, and again you must know which artifact your lens invents.

**One caveat on scope.** This is synchrosqueezing-lite. A full implementation reassigns in time as well as frequency, and genuine synchrosqueezing is *invertible*: components can be separated and reconstructed individually, which is its main practical advantage. Neither is implemented here.

---
### 🕐 Session 3 of 3 — *Empirical Mode Decomposition* (~35 min)
**Goal:** no basis at all: sift the signal into its own intrinsic oscillations.
**Builds on:** Session 2.

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: Empirical Mode Decomposition</b></summary>

**Timing (~35 min).** 8 min the sifting idea · 8 min the algorithm · 10 min the demo and why it succeeds so completely · 9 min fragility, which is the session's real content.

**Board first — place EMD on the spectrum of assumptions.** Fourier imposes sinusoids; wavelets impose a scale family; EMD imposes *nothing* and lets the data choose its own components. Write the sifting loop as three steps: spline the maxima, spline the minima, subtract their mean, repeat until what remains oscillates symmetrically about zero. That object is an intrinsic mode function, peel it off, recurse on the remainder. Students can follow the whole algorithm in five minutes, which is unusual and worth exploiting.

**Then the honest framing that should shape the whole session.** EMD is defined by an *algorithm*, not by a mathematical property. There is no basis, no uniqueness theorem, no convergence proof, and no clean way to say what an IMF is other than "what this procedure produces." That is a real difference in kind from everything else in this curriculum, and the room should feel it. It does not make EMD useless — it is genuinely effective on nonstationary, nonlinear data where the alternatives impose the wrong structure — but every claim about it is empirical.

**The demo works too well; say so before running it.** All three correlations come out at 1.000. Ask what that should make them suspect. The planted components are separated by nearly an octave-decade (60 Hz against 7 Hz), noiseless, and stationary in amplitude — a near-perfect case built to demonstrate the mechanism. A correlation of 1.000 to three decimals reports the ceiling of the test, not the robustness of the method. Getting students to distrust their own clean results is more valuable here than the decomposition itself.

**Mode mixing is the failure to name, and it is the standard one.** Bring the two frequencies closer — try 60 Hz and 25 Hz — or add a burst that exists only in part of the record, and a single IMF will start containing pieces of both components while a component gets split across two IMFs. The decomposition is then uninterpretable, and worse, it changes discontinuously with small perturbations of the input. This is why **ensemble EMD** exists: run EMD many times with different added noise and average the IMFs, which stabilises the result at considerable computational cost. Adding noise to fix an algorithm is a striking enough idea to be worth the two minutes.

**Ask the room.** "EMD assumes no basis. Is that strictly better?" No — it is a different trade. A fixed basis gives uniqueness, linearity, an inverse, and theorems; adaptivity gives components that match the data when no fixed basis would. You buy flexibility with every guarantee. That framing connects to Session 1 and 2's artifacts: each of the three methods in this workshop assumes something different and pays for it differently.

**If the demo misbehaves.** The sifting loop is short and unguarded — with a noisy or short input, `argrelextrema` can find too few extrema and the loop breaks early, returning a partial decomposition without complaint. That silent degradation is itself characteristic of EMD in practice, and worth showing if someone asks why the output has fewer IMFs than expected.
</details>

## 4. Let the Signal Choose

💡 **Intuition.** Fourier imposes sinusoids; wavelets impose scales. **EMD** imposes nothing: repeatedly *sift* — fit envelopes through the maxima and minima, subtract their mean — until what remains oscillates symmetrically (an *intrinsic mode function*), then peel it off and repeat. Nonlinear, adaptive, basis-free — and correspondingly fragile (mode mixing, no clean theory; ensemble-EMD patches it with noise). Powerful on nonstationary, nonlinear data; use with eyes open.

In [4]:
def emd(x_in, max_imfs=4, sift_iters=8):
    imfs, resid = [], x_in.astype(float).copy()
    for _ in range(max_imfs):
        h = resid.copy()
        for _ in range(sift_iters):
            maxima = sig.argrelextrema(h, np.greater)[0]
            minima = sig.argrelextrema(h, np.less)[0]
            if len(maxima) < 4 or len(minima) < 4: break
            from scipy.interpolate import CubicSpline
            upper = CubicSpline(maxima, h[maxima], bc_type="natural")(np.arange(len(h)))
            lower = CubicSpline(minima, h[minima], bc_type="natural")(np.arange(len(h)))
            h = h - (upper + lower)/2
        imfs.append(h); resid = resid - h
        if len(sig.argrelextrema(resid, np.greater)[0]) < 4: break
    return imfs, resid

# planted two-scale signal: fast oscillation + slow oscillation + trend — can EMD unmix it?
t2 = np.linspace(0, 1, 2000)
fast, slow, trend = np.sin(2*np.pi*60*t2), 0.8*np.sin(2*np.pi*7*t2), 1.5*t2**2
imfs, resid = emd(fast + slow + trend)

fig, axes = plt.subplots(len(imfs)+1, 1, figsize=(9, 1.2*(len(imfs)+1)), sharex=True)
for ax, imf in zip(axes, imfs): ax.plot(t2, imf, linewidth=0.6)
axes[-1].plot(t2, resid, linewidth=0.8, color="k"); axes[-1].set_title("residual (the trend)", fontsize=8)
plt.suptitle("EMD sifts: fast IMF, slow IMF, trend — no basis was specified")
plt.tight_layout(); plt.show()

print(f"IMF1 vs planted fast: |corr| = {abs(np.corrcoef(imfs[0], fast)[0,1]):.3f}")
print(f"IMF2 vs planted slow: |corr| = {abs(np.corrcoef(imfs[1], slow)[0,1]):.3f}")
print(f"residual vs trend:    |corr| = {abs(np.corrcoef(resid, trend)[0,1]):.3f}")

IMF1 vs planted fast: |corr| = 1.000
IMF2 vs planted slow: |corr| = 1.000
residual vs trend:    |corr| = 1.000


/tmp/ipykernel_2986310/3626029590.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Three correlations of **1.000** — IMF1 against the planted 60 Hz oscillation, IMF2 against the 7 Hz one, residual against the quadratic trend. The algorithm was told nothing: no basis, no frequencies, no number of components. It splined the maxima and minima, subtracted their mean, and the planted structure fell out.

**But three correlations of exactly 1.000 should provoke scepticism, not applause.** That is the ceiling of the test, and it reports how easy the problem was rather than how good the method is. Look at what was planted: 60 Hz against 7 Hz is nearly a decade of separation, there is no noise at all, and both components have constant amplitude across the whole record. Sifting separates well-spaced, clean oscillations essentially perfectly, and this signal was built to be exactly that. As in Session 2's `0.0 Hz`, a saturated metric is measuring its own limits.

**The genuine result underneath is still worth having.** No basis was specified and the decomposition is *adaptive* — the components emerged from the data's own extrema rather than from a dictionary chosen in advance. Fourier would have described the trend as a spray of low-frequency coefficients; wavelets would have imposed a dyadic scale family. EMD imposed nothing, and on a signal whose parts genuinely are separate oscillations, that is the right stance.

**Now the fragility, which is the real content of this session.** EMD is defined by an *algorithm*, not by a mathematical property. There is no basis, no uniqueness theorem, no convergence proof, and no definition of an IMF beyond "what this procedure produces." Everything asserted about it is empirical, which is a genuine difference in kind from the Fourier and wavelet machinery elsewhere in this curriculum.

The characteristic failure is **mode mixing**. Move the frequencies closer — 60 Hz against 25 Hz — or make one component intermittent, present in only part of the record, and a single IMF starts carrying pieces of both while one true component splits across two IMFs. The decomposition becomes uninterpretable, and it changes discontinuously under small perturbations of the input, so two nearly identical recordings can decompose completely differently. That is worth trying: it takes one edit and it is more instructive than the clean run.

The standard repair is **ensemble EMD** — run the decomposition many times with different noise added, then average the IMFs. Adding noise to stabilise an algorithm is counterintuitive enough to remember, and it works because the noise gives the sifting a consistent scale structure to latch onto. It also multiplies the cost.

**The trade, stated once.** A fixed basis gives you uniqueness, linearity, an inverse, and theorems. Adaptivity gives you components shaped by the data when no fixed basis fits. You cannot have both, and EMD sits at the far adaptive end — powerful on nonstationary, nonlinear data, and to be used with eyes open. That completes the workshop's theme: the WVD invents ghosts, reassignment smears where components cross, and EMD invents mode mixing. Every lens has its own artifacts, and reading a time–frequency picture means knowing which ones to discount.

## 5. Conclusion

Quadratic distributions buy razor concentration at the price of ghosts; synchrosqueezing spends the STFT's phase to sharpen without them (ridge width measured); EMD abandons bases entirely and recovers planted components (correlations verified) — with fragility as the tax on adaptivity. Choose the lens per signal, and always know what artifacts your lens invents.

---
## Where next

- [Audio & Speech DSP](./Audio_Speech_DSP.ipynb) — apply all three to real sound.
- [Cyclostationary Analysis](./Cyclostationary_HOS.ipynb) — a different generalization: periodicity in the *statistics*.